# Live Personal VAD demo

Run top to bottom. Cell **2** (enroll) is needed once per target person.
Cell **3** runs live until you press the kernel **interrupt** (stop) button.

## 1. Setup — load the frozen backbones + trained head

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from src.demo.live_demo_example import StreamingDetector, run_enroll, run_mic, run_wav, DEFAULT_TARGET
from src.eval.eval_model_example import load_model
from src.features.extract_features_example import load_embedder, load_vad_model

embedder = load_embedder()
vad_model = load_vad_model()

MODEL = Path("data/models/personal_vad_noisy_trailing.pt")
if not MODEL.exists():
    MODEL = Path("data/models/personal_vad_noisy.pt")
    print(f"trailing model not found -> falling back to {MODEL} "
          f"(centered-trained: expect ~1s lag; train the trailing model!)")
model = load_model(MODEL)
print("ready")

trailing model not found -> falling back to data/models/personal_vad_noisy.pt (centered-trained: expect ~1s lag; train the trailing model!)
ready


Using cache found in /Users/ihsanbolum/.cache/torch/hub/snakers4_silero-vad_master


## 2. Enroll the target (run once — records ~20 s from the mic)

In [2]:
run_enroll(seconds=20, out_path=DEFAULT_TARGET, embedder=embedder, vad_model=vad_model)

recording...
saved /Users/ihsanbolum/Target Person Voice/data/enroll/e_target.npz  (audio kept at /Users/ihsanbolum/Target Person Voice/data/enroll/e_target.wav)


## 3. Live detection

Green dot = target speaking alone. ~1.5 s warm-up, ~0.5 s switching delay.
Stop with the kernel interrupt button.

In [5]:
e_target = np.load(DEFAULT_TARGET)["e_target"]
detector = StreamingDetector(model, embedder, vad_model, e_target)
run_mic(detector)

listening... (dot needs ~1.5s of audio to warm up; Ctrl-C to stop)
   141.2s  ○ ---------------  class=other-only  cos=+0.15  target-frames=  0%   
stopped.
